# Replot Pre-calculated Coordinates

Use this notebook to reload precomputed UMAP or PCA coordinates from a `.csv`, choose a color palette, compute clusters, and visualize the result with both Matplotlib and interactive Bokeh.

In [4]:
import os
from pathlib import Path

import pandas as pd
from scipy.cluster.hierarchy import linkage, fcluster

import matplotlib.pyplot as plt
from matplotlib import colors as mcolors
from matplotlib.cm import get_cmap

from bokeh.io import output_notebook, show
from bokeh.models import ColumnDataSource, HoverTool
from bokeh.palettes import Category10, Category20, Viridis256
from bokeh.plotting import figure

output_notebook()


Loading BokehJS ...

In [1]:
# Edit these settings.
CSV_PATH = Path("/Users/eriktrebilcock/Downloads/md_michael_acceptor_UMAP_coords.csv")
ALGORITHM = "UMAP"   # "UMAP", or "PCA"
N_CLUSTERS = 11
COLOR_PALETTE = "tab20"   # e.g. "tab20", "Set2", "viridis", or "#1f77b4,#ff7f0e,#2ca02c"
MATPLOTLIB_OUT = Path("/Users/eriktrebilcock/Downloads/replot_notebook_matplotlib.png")
BOKEH_OUT = Path("/Users/eriktrebilcock/Downloads/replot_notebook_bokeh.html")


NameError: name 'Path' is not defined

In [2]:
def detect_coordinate_columns(df: pd.DataFrame, algorithm: str) -> tuple[str, str]:
    algorithm = algorithm.upper()
    candidate_pairs = [
        (f"{algorithm}1", f"{algorithm}2"),
        ("UMAP1", "UMAP2"),
        ("PCA1", "PCA2"),
        ("TSNE1", "TSNE2"),
    ]

    for col1, col2 in candidate_pairs:
        if col1 in df.columns and col2 in df.columns:
            return col1, col2

    numeric_cols = df.select_dtypes(include="number").columns.tolist()
    if len(numeric_cols) >= 2:
        return numeric_cols[0], numeric_cols[1]

    raise ValueError(
        "Could not find two coordinate columns. Expected columns such as "
        f"'{algorithm}1'/'{algorithm}2', 'UMAP1'/'UMAP2', or at least two numeric columns."
    )


def build_palette(palette_spec: str, n_colors: int) -> list:
    colors = [color.strip() for color in palette_spec.split(",") if color.strip()]
    if len(colors) > 1:
        invalid = [color for color in colors if not mcolors.is_color_like(color)]
        if invalid:
            raise ValueError(f"Invalid color values in palette list: {invalid}")
        return [colors[i % len(colors)] for i in range(n_colors)]

    cmap = get_cmap(palette_spec, n_colors)
    return [mcolors.to_hex(cmap(i)) for i in range(n_colors)]


def compute_clusters(df: pd.DataFrame, x_col: str, y_col: str, n_clusters: int) -> pd.DataFrame:
    coords = df[[x_col, y_col]].apply(pd.to_numeric, errors="coerce")
    clean = df.loc[coords.notna().all(axis=1)].copy()
    coords = coords.loc[clean.index]

    if len(clean) == 0:
        raise ValueError("No valid coordinate rows were found in the CSV.")

    if len(clean) == 1:
        clean["Cluster"] = 1
        return clean

    effective_clusters = min(n_clusters, len(clean))
    linkage_matrix = linkage(coords.to_numpy(), method="ward")
    clean["Cluster"] = fcluster(linkage_matrix, t=effective_clusters, criterion="maxclust")
    return clean


def bokeh_palette(colors: list) -> list:
    if all(color.startswith("#") for color in colors):
        return colors
    if len(colors) <= 10:
        return Category10[max(3, len(colors))][:len(colors)]
    if len(colors) <= 20:
        return Category20[20][:len(colors)]
    step = max(1, len(Viridis256) // len(colors))
    return Viridis256[::step][:len(colors)]


NameError: name 'pd' is not defined

In [ ]:
if not CSV_PATH.exists():
    raise FileNotFoundError(f"CSV not found at: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)
x_col, y_col = detect_coordinate_columns(df, ALGORITHM)
plot_df = compute_clusters(df, x_col, y_col, N_CLUSTERS)
palette = build_palette(COLOR_PALETTE, plot_df["Cluster"].nunique())

print("Loaded columns:", df.columns.tolist())
print(f"Using coordinate columns: {x_col}, {y_col}")
print(f"Clusters plotted: {plot_df['Cluster'].nunique()}")
plot_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
clusters = sorted(plot_df["Cluster"].unique())

for color, cluster_id in zip(palette, clusters):
    mask = plot_df["Cluster"] == cluster_id
    ax.scatter(
        plot_df.loc[mask, x_col],
        plot_df.loc[mask, y_col],
        s=35,
        c=color,
        edgecolors="black",
        linewidths=0.3,
        alpha=0.85,
        label=f"Cluster {cluster_id}",
    )

ax.set_xlabel(x_col)
ax.set_ylabel(y_col)
ax.set_title(f"{ALGORITHM.upper()} replot with {len(clusters)} clusters")
ax.legend(loc="best", fontsize=8, frameon=False)
plt.tight_layout()

MATPLOTLIB_OUT.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(MATPLOTLIB_OUT, dpi=300)
plt.show()
print(f"Saved Matplotlib figure to: {MATPLOTLIB_OUT}")


In [ ]:
bokeh_colors = bokeh_palette(palette)
cluster_to_color = {
    cluster_id: bokeh_colors[i]
    for i, cluster_id in enumerate(sorted(plot_df["Cluster"].unique()))
}

bokeh_df = plot_df.copy()
bokeh_df["color"] = bokeh_df["Cluster"].map(cluster_to_color)
bokeh_df["label"] = bokeh_df["Cluster"].map(lambda value: f"Cluster {value}")

source = ColumnDataSource(bokeh_df)
tooltips = [(x_col, f"@{{{x_col}}}"), (y_col, f"@{{{y_col}}}"), ("Cluster", "@Cluster")]

fig = figure(
    width=700,
    height=700,
    title=f"{ALGORITHM.upper()} replot (interactive)",
    tools="pan,wheel_zoom,box_zoom,reset,save,hover",
)
fig.add_tools(HoverTool(tooltips=tooltips))
fig.circle(
    x=x_col,
    y=y_col,
    source=source,
    size=8,
    color="color",
    alpha=0.85,
    line_color="black",
    legend_field="label",
)
fig.xaxis.axis_label = x_col
fig.yaxis.axis_label = y_col
fig.legend.location = "top_left"
fig.legend.click_policy = "hide"

show(fig)


In [ ]:
from bokeh.io import output_file, save

BOKEH_OUT.parent.mkdir(parents=True, exist_ok=True)
output_file(BOKEH_OUT)
save(fig)
print(f"Saved Bokeh HTML to: {BOKEH_OUT}")
